# CRWN Fit-Aware — Train the Measurement Head

End-to-end runnable Colab notebook that produces a defensible thesis number for the **"hand-coded coefficients vs. Keras-learned head"** comparison described in [EVALUATION.md §5](https://github.com/OmarJesse/crwn-clothing-fe/blob/main/EVALUATION.md).

**What this notebook does:**
1. Synthesizes an ANSUR II-style population (6 000 subjects) with realistic anthropometric joint distributions.
2. Projects each subject to a synthetic 17-keypoint MoveNet COCO layout (frontal view, with noise).
3. Trains the 128→64→32→6 Keras MLP for 40 epochs.
4. Evaluates **per-measurement MAE / RMSE** on a held-out 20 % split.
5. Compares against the **hand-coded baseline** copied verbatim from the deployed FE inference.
6. Exports the trained model to TensorFlow.js so it can drop into the runtime measurement pipeline.

**Expected output:** `learned MAE` should beat `baseline MAE` on every measurement except `weightKg` (where the baseline already collapses to BMI-from-height which is essentially the optimum given no weight signal in the pose).

**Total runtime on a T4 Colab GPU:** ~3 minutes.

## 1. Setup

In [ ]:
!pip install -q tensorflow==2.15.0 tensorflowjs==4.18.0 scikit-learn

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import tensorflow as tf
print(f'TF {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Pull the training script + run

Clones the FE repo so the canonical script + helpers stay in sync with the deployed pipeline.

In [ ]:
!git clone --depth 1 https://github.com/OmarJesse/crwn-clothing-fe.git /tmp/crwn-fe
%cd /tmp/crwn-fe/evaluation/python

In [ ]:
!python train_measurement_head.py --samples 8000 --epochs 50 --batch-size 64 --save results/measurement_head.h5

## 3. Inspect the results JSON

In [ ]:
import json, pandas as pd
r = json.load(open('results/measurement_head.json'))
print(f"Δ overall mean MAE (learned vs baseline): {r['mae_improvement_cm']:+.2f} cm")
rows = []
for label, base, learned in zip(r['labels'], r['baseline_mae'], r['learned_mae']):
    rows.append({'measurement': label, 'baseline_MAE': base, 'learned_MAE': learned, 'delta': base - learned})
pd.DataFrame(rows)

## 4. Convert to TF.js and drop into the FE bundle

After conversion you copy the resulting folder into `crwn-clothing-fe/public/models/measurement-head/` and swap the runtime call in [`crwn-clothing-fe/src/routes/onboarding/inference/measurements.js`](https://github.com/OmarJesse/crwn-clothing-fe/blob/main/src/routes/onboarding/inference/measurements.js) to load it via `tf.loadLayersModel('/models/measurement-head/model.json')`.

In [ ]:
!tensorflowjs_converter --input_format=keras results/measurement_head.h5 results/tfjs/measurement-head
!ls -la results/tfjs/measurement-head

In [ ]:
# Zip and download to your machine.
!cd results/tfjs && zip -r measurement-head.zip measurement-head
from google.colab import files
files.download('results/tfjs/measurement-head.zip')